####Requirement
1. Read raw data from flight_time_raw table
2. Apply transformations to time values as hour to minute interval

    1. CRS_DEP_TIME
    2. DEP_TIME
    3. WHEELS_ON
    4. CRS_ARR_TIME
    5. ARR_TIME
3. Apply transformation to TAXI_IN to make it a minute interval

In [0]:
flight_time_df = spark.read.table('dev.spark_db.flight_time_raw')

In [0]:
flight_time_df.display()

2. Convert CRS_DEP_TIME to INTERVAL HOUR TO MINUTE DayTimeIntervalType

In [0]:
from pyspark.sql.functions import lpad, col,left,expr,lit,right,concat


flight_time_df_hour_min_column_split = (
    flight_time_df.withColumns
    (
        {"CRS_DEP_TIME_interval_hour" : 
          left(lpad(col("CRS_DEP_TIME"), 4, '0'),lit(2)),
          "CRS_DEP_TIME_interval_min":
        right(lpad(col("CRS_DEP_TIME"), 4, '0'),lit(2)),

        
      
           
        }
)
)

In [0]:
 flight_time_df_daytimeintervaltype = (
     
    flight_time_df_hour_min_column_split.withColumn(

        "CRS_DEP_TIME_in_hr_min" ,
           expr(
               """CAST(
                    concat(
                        CRS_DEP_TIME_interval_hour, 
                        ':',
                        CRS_DEP_TIME_interval_min)AS
               INTERVAL HOUR TO MINUTE)""")

     )
 )
     




In [0]:
def convert_time_to_interval_hr_min(col_name):
    
    from pyspark.sql.functions import lpad, col,left,expr,lit,right,concat

    return expr(f"""
            CAST( 
                 CONCAT
                    (  
                      LEFT(LPAD({col_name}, 4, '0'),2),
                        ':',
                        RIGHT(LPAD({col_name}, 4, '0'),2)
                            
                     ) AS INTERVAL HOUR TO MINUTE
            )    
                """
               
               )

In [0]:
flight_time_df_daytimeintervaltype_updated =(
     
 flight_time_df.withColumns(
      {

    "CRS_DEP_TIME_IN_HR_MIN": convert_time_to_interval_hr_min('CRS_DEP_TIME'),

    "DEP_TIME_TIME_IN_HR_MIN": convert_time_to_interval_hr_min('DEP_TIME'),

    "CRS_ARRIVAL_TIME_IN_HR_MIN": convert_time_to_interval_hr_min('CRS_ARR_TIME'),

    "ARR_TIME_IN_HR_MIN": convert_time_to_interval_hr_min('ARR_TIME'),

    "WHEELS_ON_IN_HR_MIN": convert_time_to_interval_hr_min('WHEELS_ON'),

    "TAXI_IN_HR_MIN" : expr('CAST(TAXI_IN AS INTERVAL MINUTE)')

      }
  )

 )

In [0]:
flight_time_df_daytimeintervaltype_updated.display()

In [0]:
flight_time_df_daytimeintervaltype_updated.write.mode('overwrite').saveAsTable('dev.spark_db.flight_time')

In [0]:
%sql
SELECT * FROM dev.spark_db.flight_time